# Project: Variational inference for Bayesian logistic regression 🎲
MCMC gives exact samples but is slow; **variational inference** turns Bayesian inference into optimisation: pick a simple family $q(w)=\mathcal N(\mu, \text{diag}(\sigma^2))$ and maximise the **ELBO** with the reparameterisation trick and Adam. Compare against a long Metropolis run, and see VI's famous habit of underestimating uncertainty when parameters are correlated.

Topic: [[Variational Inference]], [[Bayesian Inference]], [[MCMC]] · guide note: [[Project - Variational Inference for Logistic Regression]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn.functional as F
torch.manual_seed(0)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
rng = np.random.default_rng(0); n = 60
x1 = rng.normal(0, 1, n); x2 = x1 + rng.normal(0, 0.3, n)                         # two strongly correlated features
X_np = np.c_[np.ones(n), x1, x2]; w_true = np.array([0.5, 1.0, 1.0])
y_np = (rng.random(n) < 1 / (1 + np.exp(-X_np @ w_true))).astype(float)
X, y = torch.tensor(X_np, dtype=torch.float32), torch.tensor(y_np, dtype=torch.float32)
PRIOR_VAR = 10.0
def log_post_np(w): z = X_np @ w; return float(np.sum(y_np * z - np.logaddexp(0, z)) - 0.5 * w @ w / PRIOR_VAR)
print(f"{n} examples, corr(x1, x2) = {np.corrcoef(x1, x2)[0, 1]:.2f}")

## 1. The ELBO with the reparameterisation trick
$\text{ELBO}(\mu,\sigma) = \mathbb E_{q}[\log p(y\mid X,w)] + \mathbb E_q[\log p(w)] + H[q]$. Sample $w = \mu + \sigma\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$ (`n_samples` rows) so gradients flow to $\mu$ and $\log\sigma$. Log-likelihood: $-\text{BCE}$ summed over data, averaged over samples. Log-prior: $-\frac{\lVert w\rVert^2}{2\cdot\text{prior\_var}}$ averaged over samples. Entropy of a diagonal Gaussian: $\sum_j\log\sigma_j$ (+ a constant we can drop).

In [ ]:
def elbo(mu, log_sigma, X, y, prior_var=PRIOR_VAR, n_samples=32):
    # TODO 1: reparameterised Monte Carlo estimate of the ELBO (up to a constant)
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_elbo():
    torch.manual_seed(0); mu = torch.zeros(3, requires_grad=True); ls = torch.full((3,), -20.0, requires_grad=True)
    e = elbo(mu, ls, X, y, n_samples=4); e.backward()
    exact = -n * np.log(2) - 60.0                     # σ≈0: w = 0 → log-lik −n·log 2, prior 0, entropy Σ log σ = −60
    return abs(e.item() - exact) < 1e-3 and mu.grad is not None and ls.grad is not None
check("ELBO", _t_elbo, "With σ ≈ 0 at μ = 0 the ELBO must equal −n·log 2 + Σ log σ, and gradients must reach μ and log σ.")
if ready("ELBO"):
    mu = torch.zeros(3, requires_grad=True); log_sigma = torch.full((3,), -1.0, requires_grad=True); opt = torch.optim.Adam([mu, log_sigma], lr=0.03); hist = []
    for step in range(3000):
        loss = -elbo(mu, log_sigma, X, y); opt.zero_grad(); loss.backward(); opt.step(); hist.append(-loss.item())
    vi_mu, vi_sd = mu.detach().numpy(), log_sigma.exp().detach().numpy()
    r = np.random.default_rng(1); w = np.zeros(3); lp = log_post_np(w); samples = []                           # long Metropolis run as "ground truth"
    for i in range(60000):
        prop = w + r.normal(0, 0.25, 3); lpp = log_post_np(prop)
        if np.log(r.random()) < lpp - lp: w, lp = prop, lpp
        if i >= 10000: samples.append(w.copy())
    S = np.array(samples); mc_mu, mc_sd = S.mean(0), S.std(0)
    for j, name in enumerate(["bias", "w1", "w2"]): print(f"{name:<4} mean: VI {vi_mu[j]:+.2f} MCMC {mc_mu[j]:+.2f} | std: VI {vi_sd[j]:.2f} MCMC {mc_sd[j]:.2f}")
    print(f"posterior correlation of w1 and w2 (MCMC): {np.corrcoef(S[:, 1], S[:, 2])[0, 1]:+.2f} – mean-field VI assumes 0")
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.6)); ax[0].plot(hist); ax[0].set_title("ELBO"); ax[0].set_ylim(np.percentile(hist, 5), max(hist) + 2)
    ax[1].scatter(S[::20, 1], S[::20, 2], s=2, alpha=.3, label="MCMC samples")
    vs = np.random.default_rng(2).normal(vi_mu, vi_sd, (1000, 3)); ax[1].scatter(vs[:, 1], vs[:, 2], s=2, alpha=.3, label="VI samples"); ax[1].set_xlabel("w1"); ax[1].set_ylabel("w2"); ax[1].legend(); plt.show()
    check("VI vs MCMC", lambda: np.all(np.abs(vi_mu - mc_mu) < 0.5) and vi_sd[1] < 0.8 * mc_sd[1] and vi_sd[2] < 0.8 * mc_sd[2],
          "VI means should be close to MCMC's, and its std for the correlated weights clearly smaller (mean-field underestimates).")

<details><summary>▶ Solution</summary>

```python
def elbo(mu, log_sigma, X, y, prior_var=PRIOR_VAR, n_samples=32):
    eps = torch.randn(n_samples, len(mu))
    w = mu + torch.exp(log_sigma) * eps
    logits = X @ w.T
    log_lik = -F.binary_cross_entropy_with_logits(logits, y[:, None].expand_as(logits), reduction="none").sum(0).mean()
    log_prior = (-0.5 * (w ** 2).sum(1) / prior_var).mean()
    entropy = log_sigma.sum()
    return log_lik + log_prior + entropy
```
</details>

## Stretch goals
- Full-covariance VI: $w = \mu + L\epsilon$ with a lower-triangular $L$. Does the variance underestimation disappear?
- Remove the correlation (make x2 independent) and compare again.
- Use the VI posterior for predictions: average σ(xᵀw) over samples vs. plugging in μ.